# CS781 Capstone · Phase 2 · Legal NER (single notebook)

In [ ]:
import os, sys, subprocess, json, glob, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "huggingface_hub"], check=False)
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
os.makedirs(f"{WORK}/p2lner/phase1_models", exist_ok=True)
os.chdir(WORK); sys.path.insert(0, WORK)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable GPU T4!)")

## 2. Pipeline code (written to `p2lner/`)

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/common.py
"""Shared building blocks (RMSNorm, RoPE, SwiGLU, init).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (identical in A/B/C/D notebooks, cell 19),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def _ffn_hidden(d_model, mult, multiple_of=64):
    h = int(mult * d_model)
    return multiple_of * ((h + multiple_of - 1) // multiple_of)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head_dim"
        inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        T = x.size(-2)
        assert pos0 + T <= self.cos.size(0), f"position {pos0 + T} > RoPE table {self.cos.size(0)}"
        cos, sin = self.cos[pos0:pos0 + T], self.sin[pos0:pos0 + T]
        x1, x2 = x.float().chunk(2, dim=-1)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden, dropout=0.0, bias=False):
        super().__init__()
        self.w12 = nn.Linear(d_model, 2 * hidden, bias=bias)
        self.proj = nn.Linear(hidden, d_model, bias=bias)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, b = self.w12(x).chunk(2, dim=-1)
        return self.drop(self.proj(F.silu(a) * b))

def init_weights_(model, n_layer):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

def lm_loss(logits, targets):
    return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/arch_a.py
"""Architecture A: decoder-only LLaMA-style (GQA, RoPE, SwiGLU).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (decoder-only-fine-tuned.ipynb / decoder-only-transformer-llama-style.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class GQAttention(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0 and n_head % n_kv_head == 0
        self.n_head, self.n_kv, self.hd = n_head, n_kv_head, d_model // n_head
        self.wq = nn.Linear(d_model, n_head * self.hd, bias=bias)
        self.wkv = nn.Linear(d_model, 2 * n_kv_head * self.hd, bias=bias)
        self.proj = nn.Linear(n_head * self.hd, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, pos0=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        k, v = self.wkv(x).view(B, T, 2, self.n_kv, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        if self.n_kv != self.n_head:
            rep = self.n_head // self.n_kv
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, -1)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_head, n_kv_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, pos0, cache)
        return x + self.ffn(self.norm2(x))

class LlamaLM(nn.Module):
    def __init__(self, vocab_size, block_size, pad_id=None, n_layer=12, n_head=10, n_kv_head=2, d_model=640,
                 ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[LlamaLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.pad_id, self.n_layer = block_size, pad_id, n_layer
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, block_size, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, n_kv_head, hidden, dropout, bias) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, pos0=0, caches=None, last_only=False):
        B, T = idx.shape
        assert pos0 + T <= self.block_size, f"position {pos0 + T} > block_size {self.block_size}"
        x = self.drop(self.tok_emb(idx))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = LlamaLM

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/arch_b.py
"""Architecture B: encoder-decoder.

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (B/notebookc976077ad3.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.hd = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, pos0=0, key_mask=None, causal=False, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        mask = None if key_mask is None else key_mask[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and T > 1 and mask is None,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        self.n_head, self.hd = n_head, d_model // n_head
        self.q = nn.Linear(d_model, d_model, bias=bias)
        self.kv = nn.Linear(d_model, 2 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, memory, mem_mask, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        if cache is not None and "ck" in cache:
            k, v = cache["ck"], cache["cv"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["ck"], cache["cv"] = k, v
        mask = None if mem_mask is None else mem_mask[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = SelfAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, key_mask):
        x = x + self.attn(self.norm1(x), rope, 0, key_mask=key_mask, causal=False)
        return x + self.ffn(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = RMSNorm(d_model), RMSNorm(d_model), RMSNorm(d_model)
        self.self_attn = SelfAttention(d_model, n_head, dropout, bias)
        self.cross_attn = CrossAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, memory, mem_mask, pos0=0, cache=None):
        x = x + self.self_attn(self.norm1(x), rope, pos0, causal=True, cache=cache)
        x = x + self.cross_attn(self.norm2(x), memory, mem_mask, cache=cache)
        return x + self.ffn(self.norm3(x))

class EncDecLM(nn.Module):
    def __init__(self, vocab_size, enc_block, dec_block, pad_id=None, n_enc_layer=6, n_dec_layer=6,
                 n_head=9, d_model=576, ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[EncDecLM] ignoring unknown config keys: {sorted(kw)}")
        self.enc_block, self.dec_block, self.pad_id = enc_block, dec_block, pad_id
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        hd = d_model // n_head
        self.enc_rope = RotaryEmbedding(hd, enc_block, rope_theta)
        self.dec_rope = RotaryEmbedding(hd, dec_block, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.enc_blocks = nn.ModuleList([EncoderBlock(d_model, n_head, hidden, dropout, bias) for _ in range(n_enc_layer)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(d_model, n_head, hidden, dropout, bias) for _ in range(n_dec_layer)])
        self.enc_norm, self.dec_norm = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_enc_layer + n_dec_layer)
        self.lm_head.weight = self.tok_emb.weight

    def encode(self, src, src_mask=None):
        assert src.size(1) <= self.enc_block, f"source length {src.size(1)} > enc_block {self.enc_block}"
        if src_mask is not None and bool(src_mask.all()):
            src_mask = None
        x = self.drop(self.tok_emb(src))
        for blk in self.enc_blocks:
            x = blk(x, self.enc_rope, src_mask)
        return self.enc_norm(x)

    def decode(self, dec_in, memory, mem_mask=None, pos0=0, caches=None, last_only=False):
        assert pos0 + dec_in.size(1) <= self.dec_block, f"decoder position {pos0 + dec_in.size(1)} > dec_block"
        if mem_mask is not None and bool(mem_mask.all()):
            mem_mask = None
        x = self.drop(self.tok_emb(dec_in))
        for i, blk in enumerate(self.dec_blocks):
            x = blk(x, self.dec_rope, memory, mem_mask, pos0, None if caches is None else caches[i])
        x = self.dec_norm(x)
        if last_only:
            x = x[:, -1:]
        return self.lm_head(x)

    def forward(self, src, src_mask, dec_in, targets=None):
        memory = self.encode(src, src_mask)
        logits = self.decode(dec_in, memory, src_mask)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = EncDecLM

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/arch_c.py
"""Architecture C: encoder/hybrid prefix-LM + MLM (UniLM-style).

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (C/encoder-hybrid.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class HybridSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.hd = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, causal=True, pos0=0, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None:
            if "k" in cache:
                assert T == 1, "cached decoding feeds one token at a time"
                k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                           is_causal=causal and attn_mask is None and T > 1,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, C)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = HybridSelfAttention(d_model, n_head, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, attn_mask, causal, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, attn_mask, causal, pos0, cache)
        return x + self.ffn(self.norm2(x))

class HybridLM(nn.Module):
    def __init__(self, vocab_size, block_size, pad_id=None, n_layer=10, n_head=10, d_model=640, ffn_mult=8 / 3,
                 dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, n_segments=3, **kw):
        super().__init__()
        if kw:
            print(f"[HybridLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.pad_id, self.n_layer = block_size, pad_id, n_layer
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.seg_emb = nn.Embedding(n_segments, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, block_size, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, hidden, dropout, bias) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        nn.init.zeros_(self.seg_emb.weight)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    @staticmethod
    def hybrid_mask(prefix_len, T):
        pos = torch.arange(T, device=prefix_len.device)
        in_prefix = pos[None, None, :] < prefix_len[:, None, None]
        causal = pos[None, :, None] >= pos[None, None, :]
        return (in_prefix | causal)[:, None]

    def forward(self, idx, prefix_len=None, seg=None, targets=None, pos0=0, caches=None, last_only=False):
        B, T = idx.shape
        assert pos0 + T <= self.block_size, f"position {pos0 + T} > block_size {self.block_size}"
        if prefix_len is None:
            prefix_len = torch.zeros(B, dtype=torch.long, device=idx.device)
        prefix_len = prefix_len.to(idx.device).long().clamp(0, T)
        if bool((prefix_len == 0).all()):
            attn_mask, causal = None, True
        elif bool((prefix_len == T).all()):
            attn_mask, causal = None, False
        else:
            attn_mask, causal = self.hybrid_mask(prefix_len, T), False
        if seg is None:
            seg = (torch.arange(T, device=idx.device)[None, :] >= prefix_len[:, None]).long()
        x = self.drop(self.tok_emb(idx) + self.seg_emb(seg))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, attn_mask, causal, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = HybridLM

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/arch_d.py
"""Architecture D: efficient decoder, sliding-window attention + MQA.

Copied VERBATIM from the Phase-1 notebook that produced the checkpoint (D/notebook7e00370b82.ipynb),
so parameter names match model_final.pt. Do not edit: Phase-2 code wraps these modules instead."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from .common import RMSNorm, RotaryEmbedding, SwiGLU, _ffn_hidden, init_weights_, lm_loss

class LocalMQAttention(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, window, dropout=0.0, bias=False):
        super().__init__()
        assert d_model % n_head == 0 and n_head % n_kv_head == 0
        self.n_head, self.n_kv, self.hd, self.window = n_head, n_kv_head, d_model // n_head, window
        self.wq = nn.Linear(d_model, n_head * self.hd, bias=bias)
        self.wkv = nn.Linear(d_model, 2 * n_kv_head * self.hd, bias=bias)
        self.proj = nn.Linear(n_head * self.hd, d_model, bias=bias)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)
        self._mask_cache = {}

    def _block_mask(self, nb, device):
        key = (nb, str(device))
        m = self._mask_cache.get(key)
        if m is None:
            W = self.window
            a = torch.arange(W, device=device)[:, None]
            b = torch.arange(2 * W, device=device)[None, :]
            keep = ((b > a) & (b <= a + W))[None].repeat(nb, 1, 1)
            keep[0, :, :W] = False
            m = keep[None, None].expand(1, self.n_head, nb, W, 2 * W).reshape(1, self.n_head * nb, W, 2 * W)
            self._mask_cache = {key: m}
        return m

    def _blocked(self, q, k, v):
        B, H, T, hd = q.shape
        W, Hkv = self.window, k.size(1)
        Tp = -(-T // W) * W
        if Tp != T:
            q, k, v = (F.pad(t, (0, 0, 0, Tp - T)) for t in (q, k, v))
        nb = Tp // W
        k = F.pad(k, (0, 0, W, 0)).unfold(2, 2 * W, W).transpose(-1, -2)
        v = F.pad(v, (0, 0, W, 0)).unfold(2, 2 * W, W).transpose(-1, -2)
        rep = H // Hkv
        k = k[:, :, None].expand(B, Hkv, rep, nb, 2 * W, hd).reshape(B, H * nb, 2 * W, hd)
        v = v[:, :, None].expand(B, Hkv, rep, nb, 2 * W, hd).reshape(B, H * nb, 2 * W, hd)
        qb = q.reshape(B, H * nb, W, hd)
        y = F.scaled_dot_product_attention(qb, k, v, attn_mask=self._block_mask(nb, q.device),
                                           dropout_p=self.dropout if self.training else 0.0)
        return y.reshape(B, H, Tp, hd)[:, :, :T]

    def forward(self, x, rope, pos0=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.hd).transpose(1, 2)
        k, v = self.wkv(x).view(B, T, 2, self.n_kv, self.hd).permute(2, 0, 3, 1, 4)
        q, k = rope(q, pos0), rope(k, pos0)
        if cache is not None and "k" in cache:
            assert T == 1, "cached decoding feeds one token at a time"
            k = torch.cat([cache["k"], k], 2)[:, :, -self.window:]
            v = torch.cat([cache["v"], v], 2)[:, :, -self.window:]
            cache["k"], cache["v"] = k, v
            rep = self.n_head // self.n_kv
            y = F.scaled_dot_product_attention(q, k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1))
        else:
            y = self._blocked(q, k, v)
            if cache is not None:
                cache["k"], cache["v"] = k[:, :, -self.window:], v[:, :, -self.window:]
        return self.resid_drop(self.proj(y.transpose(1, 2).reshape(B, T, -1)))

class Block(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, window, ffn_hidden, dropout, bias):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = LocalMQAttention(d_model, n_head, n_kv_head, window, dropout, bias)
        self.ffn = SwiGLU(d_model, ffn_hidden, dropout, bias)

    def forward(self, x, rope, pos0=0, cache=None):
        x = x + self.attn(self.norm1(x), rope, pos0, cache)
        return x + self.ffn(self.norm2(x))

class EfficientLM(nn.Module):
    def __init__(self, vocab_size, block_size, window=256, pad_id=None, n_layer=12, n_head=10, n_kv_head=1,
                 d_model=640, ffn_mult=8 / 3, dropout=0.0, rope_theta=10000.0, rope_max_len=8192,
                 tie_weights=True, bias=False, **kw):
        super().__init__()
        if kw:
            print(f"[EfficientLM] ignoring unknown config keys: {sorted(kw)}")
        self.block_size, self.window, self.pad_id, self.n_layer = block_size, window, pad_id, n_layer
        self.max_pos = max(rope_max_len, block_size)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_head, self.max_pos, rope_theta)
        hidden = _ffn_hidden(d_model, ffn_mult)
        self.blocks = nn.ModuleList([Block(d_model, n_head, n_kv_head, window, hidden, dropout, bias)
                                     for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        init_weights_(self, n_layer)
        if tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, pos0=0, caches=None, last_only=False):
        x = self.drop(self.tok_emb(idx))
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope, pos0, None if caches is None else caches[i])
        x = self.norm_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else lm_loss(logits, targets)
        return logits, loss

MODEL_CLASS = EfficientLM

In [ ]:
%%writefile {WORK}/p2lner/phase1_models/__init__.py
from .arch_a import LlamaLM
from .arch_b import EncDecLM
from .arch_c import HybridLM
from .arch_d import EfficientLM

MODEL_CLASSES = {c.__name__: c for c in (LlamaLM, EncDecLM, HybridLM, EfficientLM)}

In [ ]:
%%writefile {WORK}/p2lner/__init__.py
"""CS781 Capstone Phase 2: Legal NER on top of the Phase-1 legal language models."""

In [ ]:
%%writefile {WORK}/p2lner/tokenizer.py
"""Byte-level BPE tokenizer from Phase 1 (load-only copy) + character offsets for NER.

Phase 1 trained the merges; here we only *load* them (same ids as pretraining) and add
`encode_with_offsets`, which returns for every token the character span it covers in the original text.

Offsets are computed per pre-token: the regex gives each pre-token's char span, BPE splits the pre-token's
UTF-8 bytes into tokens, and a byte->char table maps every token's byte range back to characters.
"""
import json
from dataclasses import dataclass

import regex as re_u

SPECIAL_TOKENS = ["<|endoftext|>", "<|pad|>", "<|bos|>", "<|sep|>", "<|mask|>"]

FINE_PUNCT_PATTERN = (r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}{1,3}| ?[^\s\p{L}\p{N}]|\s*[\r\n]+|\s+(?!\S)|\s+""")

def _merge_ids(ids, pair, new_id):
    out, i, n = [], 0, len(ids)
    a, b = pair
    while i < n:
        if i < n - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

@dataclass
class Encoded:
    ids: list
    starts: list
    ends: list

class BPETokenizer:
    def __init__(self, merges, pattern, specials=SPECIAL_TOKENS, fine_punct=False):
        self.base_pattern = pattern
        self.fine_punct = fine_punct
        self.pattern = FINE_PUNCT_PATTERN if fine_punct else pattern
        self.pat = re_u.compile(self.pattern)
        self.merges = [tuple(m) for m in merges]
        self.specials = list(specials)
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        base = 256 + len(self.merges)
        self.n_regular = base
        self.special_to_id = {s: base + i for i, s in enumerate(self.specials)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.vocab_size = base + len(self.specials)
        self.cache = {}

    @classmethod
    def load(cls, path, fine_punct=False):
        d = json.load(open(path, encoding="utf-8"))
        return cls(d["merges"], d["pattern"], d.get("specials", SPECIAL_TOKENS), fine_punct=fine_punct)

    @property
    def pad_id(self): return self.special_to_id["<|pad|>"]
    @property
    def bos_id(self): return self.special_to_id["<|bos|>"]
    @property
    def mask_id(self): return self.special_to_id["<|mask|>"]

    def _encode_word(self, s):
        hit = self.cache.get(s)
        if hit is not None:
            return hit
        ids = list(s.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best, best_r = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best, best_r = p, r
            if best is None:
                break
            ids = _merge_ids(ids, best, 256 + best_r)
        if len(self.cache) > 500_000:
            self.cache.clear()
        self.cache[s] = ids
        return ids

    def encode(self, text):
        out = []
        for w in self.pat.findall(text):
            out.extend(self._encode_word(w))
        return out

    def encode_with_offsets(self, text):
        ids, starts, ends = [], [], []
        for m in self.pat.finditer(text):
            w, c0 = m.group(), m.start()
            wid = self._encode_word(w)
            if len(wid) == 1:
                ids.append(wid[0]); starts.append(c0); ends.append(m.end())
                continue
            b2c = []
            for ci, ch in enumerate(w):
                b2c.extend([ci] * len(ch.encode("utf-8")))
            b = 0
            for t in wid:
                nb = len(self.vocab[t])
                ids.append(t)
                starts.append(c0 + b2c[b])
                ends.append(c0 + b2c[b + nb - 1] + 1)
                b += nb
        return Encoded(ids, starts, ends)

    def decode(self, ids):
        return b"".join(self.vocab[int(i)] for i in ids if int(i) < self.n_regular).decode("utf-8", errors="replace")

In [ ]:
%%writefile {WORK}/p2lner/data.py
"""Phase-2 data: download, load, validate, split.

Layout (HF dataset Exploration-Lab/CS-781-Capstone, folder Phase_2_LNER):
    Train/fold1-text.json  {doc_id: text}
    Train/fold1-gold.json  {doc_id: [[start, end, label, text], ...]}   (format auto-detected, see _parse_gold)
    Train/fold2-text.json, Train/fold2-gold.json, Train/labels.txt
    Test/fold3-text.json
"""
import json
import os
import random
from collections import Counter
from pathlib import Path

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_SUBDIR = "Phase_2_LNER"

def download(local_dir, token=None):
    from huggingface_hub import snapshot_download
    token = token or os.environ.get("HF_TOKEN")
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=local_dir, token=token,
                      allow_patterns=[f"{HF_SUBDIR}/*"])
    return Path(local_dir) / HF_SUBDIR

def find_data_dir(candidates=()):
    roots = [Path(c) for c in candidates] + [Path("/kaggle/input")]
    for r in roots:
        if not r.exists():
            continue
        if (r / "Train" / "labels.txt").exists():
            return r
        for p in r.rglob("labels.txt"):
            if p.parent.name == "Train":
                return p.parent.parent
    return None

def _parse_gold(raw, texts):
    if isinstance(raw, list):
        raw = {str(r.get("id", r.get("doc_id"))): r.get("entities", r.get("spans", r.get("annotations")))
               for r in raw}
    out, ent_text = {}, {}
    for did, ents in raw.items():
        did = str(did)
        lst, txts = [], []
        for e in ents or []:
            if isinstance(e, dict):
                s = e.get("start", e.get("start_char", e.get("begin")))
                t = e.get("end", e.get("end_char"))
                lab = e.get("label", e.get("type", e.get("labels")))
                if isinstance(lab, list):
                    lab = lab[0]
                txt = e.get("text", e.get("entity_text"))
            else:
                s, t, lab = e[0], e[1], e[2]
                txt = e[3] if len(e) > 3 else None
            lst.append((int(s), int(t), str(lab))); txts.append(txt)
        out[did] = lst; ent_text[did] = txts
    exc = inc = 0
    for did, lst in out.items():
        doc = texts.get(did, "")
        for (s, t, _), txt in zip(lst, ent_text[did]):
            if txt is None:
                continue
            exc += doc[s:t] == txt
            inc += doc[s:t + 1] == txt
    if inc > exc:
        print(f"[data] gold `end` looks INCLUSIVE ({inc} vs {exc} matches) -> converting to exclusive")
        out = {d: [(s, t + 1, l) for s, t, l in lst] for d, lst in out.items()}
    elif exc + inc:
        print(f"[data] gold `end` is exclusive: {exc}/{sum(len(v) for v in ent_text.values())} entity texts match text[s:e]")
    return out

def load_phase2(data_dir):
    data_dir = Path(data_dir)
    tr = data_dir / "Train"
    labels = [l.strip() for l in open(tr / "labels.txt", encoding="utf-8") if l.strip()]
    folds = {}
    for k in (1, 2):
        texts = {str(i): t for i, t in json.load(open(tr / f"fold{k}-text.json", encoding="utf-8")).items()}
        gold = _parse_gold(json.load(open(tr / f"fold{k}-gold.json", encoding="utf-8")), texts)
        for d in texts:
            gold.setdefault(d, [])
        folds[k] = {"texts": texts, "gold": gold}
    test = {str(i): t for i, t in json.load(open(data_dir / "Test" / "fold3-text.json", encoding="utf-8")).items()}
    return {"labels": labels, "folds": folds, "test": test}

def make_split(data, mode="random", dev_frac=0.15, seed=1337):
    pool = {}
    for k, f in data["folds"].items():
        for d, t in f["texts"].items():
            pool[d] = {"text": t, "gold": sorted(f["gold"][d]), "fold": k}
    ids = sorted(pool)
    if mode == "none":
        return pool, {}
    if mode in ("fold1", "fold2"):
        k = int(mode[-1])
        dev = [d for d in ids if pool[d]["fold"] == k]
    else:
        rng = random.Random(seed)
        rng.shuffle(ids)
        dev = ids[:max(1, round(dev_frac * len(ids)))]
    dev = set(dev)
    return ({d: v for d, v in pool.items() if d not in dev}, {d: v for d, v in pool.items() if d in dev})

def validate_gold(docs, labels):
    bad_label = bad_range = overlaps = nested = 0
    for d in docs.values():
        n = len(d["text"])
        g = sorted(d["gold"])
        for s, e, l in g:
            bad_label += l not in labels
            bad_range += not (0 <= s < e <= n)
        for (s1, e1, _), (s2, e2, _) in zip(g, g[1:]):
            if s2 < e1:
                overlaps += 1
                nested += e2 <= e1
    return {"unknown_label": bad_label, "bad_range": bad_range, "overlapping_pairs": overlaps, "nested_pairs": nested}

def eda(docs, labels, tok=None):
    lab = Counter(l for d in docs.values() for _, _, l in d["gold"])
    chars = [len(d["text"]) for d in docs.values()]
    ent_len = Counter()
    for d in docs.values():
        for s, e, l in d["gold"]:
            ent_len[l] += e - s
    rep = {"n_docs": len(docs), "n_entities": sum(lab.values()),
           "chars_per_doc": {"min": min(chars), "mean": round(sum(chars) / len(chars)), "max": max(chars)},
           "label_counts": {l: lab.get(l, 0) for l in labels},
           "mean_entity_chars": {l: round(ent_len[l] / lab[l], 1) if lab[l] else 0 for l in labels}}
    if tok is not None:
        ntok = [len(tok.encode(d["text"])) for d in docs.values()]
        rep["tokens_per_doc"] = {"min": min(ntok), "mean": round(sum(ntok) / len(ntok)), "max": max(ntok),
                                 "total": sum(ntok)}
    return rep

In [ ]:
%%writefile {WORK}/p2lner/spans.py
"""Character spans <-> token BIO tags, constrained (Viterbi) decoding, and data-driven boundary trimming.

Tag ids: 0 = O, then for label i: B = 1 + 2i, I = 2 + 2i  (12 labels -> 25 tags).

Whitespace-only tokens (e.g. "\n\n" or a run of spaces) are *transparent*: they never start or end an entity.
Their training tag is I-X if they sit inside an entity, otherwise O; at decoding time they are skipped.
A token's "content span" is its char span with leading/trailing whitespace removed (" Court" -> "Court"),
so a decoded entity starts at the first content char of its first token and ends after the last content
char of its last token.
"""
import bisect
from collections import Counter

import numpy as np

class TagSet:
    def __init__(self, labels):
        self.labels = list(labels)
        self.tags = ["O"] + [f"{p}-{l}" for l in self.labels for p in ("B", "I")]
        self.n = len(self.tags)

    def B(self, i): return 1 + 2 * i
    def I(self, i): return 2 + 2 * i
    def label_of(self, t): return None if t == 0 else self.labels[(t - 1) // 2]
    def is_B(self, t): return t > 0 and (t - 1) % 2 == 0
    def is_I(self, t): return t > 0 and (t - 1) % 2 == 1

def token_views(text, enc):
    cs, ce, ws = [], [], []
    for s, e in zip(enc.starts, enc.ends):
        t = text[s:e]
        st = t.strip()
        if not st:
            cs.append(s); ce.append(e); ws.append(True)
        else:
            cs.append(s + (len(t) - len(t.lstrip()))); ce.append(e - (len(t) - len(t.rstrip()))); ws.append(False)
    return np.array(cs), np.array(ce), np.array(ws, dtype=bool)

def spans_to_tags(text, enc, gold, tagset):
    cs, ce, ws = token_views(text, enc)
    N = len(cs)
    tags = np.zeros(N, dtype=np.int64)
    taken = np.zeros(N, dtype=bool)
    lab_idx = {l: i for i, l in enumerate(tagset.labels)}
    skipped = 0
    starts = enc.starts
    for s, e, l in sorted(gold, key=lambda x: -(x[1] - x[0])):
        if l not in lab_idx:
            continue
        lo = max(0, bisect.bisect_right(starts, s) - 1)
        hi = bisect.bisect_left(starts, e)
        idx = [i for i in range(lo, min(hi + 1, N)) if not ws[i] and cs[i] < e and ce[i] > s]
        if not idx:
            skipped += 1
            continue
        a, b = idx[0], idx[-1]
        if taken[a:b + 1].any():
            skipped += 1
            continue
        k = lab_idx[l]
        tags[a:b + 1] = tagset.I(k)
        tags[a] = tagset.B(k)
        taken[a:b + 1] = True
    return tags, skipped

def tags_to_spans(cs, ce, ws, tags, tagset):
    out = []
    cur = None
    for i in range(len(tags)):
        if ws[i]:
            continue
        t = int(tags[i])
        if t == 0:
            if cur: out.append(tuple(cur)); cur = None
            continue
        lab = tagset.label_of(t)
        if tagset.is_B(t) or cur is None or cur[2] != lab:
            if cur: out.append(tuple(cur))
            cur = [int(cs[i]), int(ce[i]), lab]
        else:
            cur[1] = int(ce[i])
    if cur:
        out.append(tuple(cur))
    return out

def bio_constraints(tagset):
    K = tagset.n
    trans = np.zeros((K, K), dtype=np.float32)
    start = np.zeros(K, dtype=np.float32)
    for j in range(K):
        if tagset.is_I(j):
            start[j] = -np.inf
            for i in range(K):
                if not (i > 0 and tagset.label_of(i) == tagset.label_of(j)):
                    trans[i, j] = -np.inf
    return start, trans

def viterbi(emis, trans, start, end=None):
    N, K = emis.shape
    if N == 0:
        return np.zeros(0, dtype=np.int64)
    score = start + emis[0]
    back = np.zeros((N, K), dtype=np.int64)
    for t in range(1, N):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + emis[t]
    if end is not None:
        score = score + end
    path = np.zeros(N, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(N - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def decode_doc(text, enc, scores, tagset, strategy="bio_viterbi", trans=None, start=None, end=None):
    cs, ce, ws = token_views(text, enc)
    if strategy == "greedy":
        tags = scores.argmax(-1)
    else:
        s0, tr0 = bio_constraints(tagset)
        tr = tr0 if trans is None else tr0 + trans
        st = s0 if start is None else s0 + start
        tags = viterbi(scores.astype(np.float32), tr, st, end)
    return tags_to_spans(cs, ce, ws, tags, tagset)

def learn_trim_rules(docs, max_rate=0.002):
    first, last, n = Counter(), Counter(), 0
    seen_chars = set()
    for d in docs.values():
        t = d["text"]
        seen_chars.update(c for c in set(t) if not c.isalnum() and not c.isspace())
        for s, e, _ in d["gold"]:
            first[t[s]] += 1; last[t[e - 1]] += 1; n += 1
    n = max(n, 1)
    lead = {c for c in seen_chars if first[c] / n <= max_rate}
    trail = {c for c in seen_chars if last[c] / n <= max_rate}
    return {"lead": "".join(sorted(lead)), "trail": "".join(sorted(trail))}

def trim_spans(text, spans, rules):
    if not rules:
        return spans
    lead, trail = set(rules["lead"]), set(rules["trail"])
    out = []
    for s, e, l in spans:
        while s < e and (text[s] in lead or text[s].isspace()):
            s += 1
        while e > s and (text[e - 1] in trail or text[e - 1].isspace()):
            e -= 1
        if e > s:
            out.append((s, e, l))
    return out

def reachability(docs, tok, tagset):
    from .metrics import strict_scores
    pred, bad, skipped = {}, [], 0
    for did, d in docs.items():
        enc = tok.encode_with_offsets(d["text"])
        tags, sk = spans_to_tags(d["text"], enc, d["gold"], tagset)
        skipped += sk
        cs, ce, ws = token_views(d["text"], enc)
        pred[did] = tags_to_spans(cs, ce, ws, tags, tagset)
        ps = set(pred[did])
        for g in d["gold"]:
            if tuple(g) not in ps:
                s, e, l = g
                bad.append((did, s, e, l, d["text"][max(0, s - 15):s] + "[" + d["text"][s:e] + "]" + d["text"][e:e + 15]))
    gold = {k: d["gold"] for k, d in docs.items()}
    return strict_scores(gold, pred, tagset.labels), bad, skipped

In [ ]:
%%writefile {WORK}/p2lner/metrics.py
"""Strict span-level metrics (exact start, end and label) + error analysis.

Macro averaging: per-label P/R/F1 averaged over the labels in labels.txt; a label that has neither gold
nor predicted entities in the evaluated documents is left out of the average. Cross-check against the
official evaluation script from the course Drive before trusting small differences.
"""
from collections import Counter, defaultdict

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = set(map(tuple, gold[did]))
        p = set(map(tuple, pred.get(did, [])))
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        if tp[l] + fp[l] + fn[l] == 0:
            continue
        p, r, f = _prf(tp[l], fp[l], fn[l])
        per[l] = {"p": p, "r": r, "f1": f, "tp": tp[l], "fp": fp[l], "fn": fn[l], "support": tp[l] + fn[l]}
    n = max(len(per), 1)
    T, F, N = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mp, mr, mf = _prf(T, F, N)
    return {"macro_f1": sum(v["f1"] for v in per.values()) / n,
            "macro_p": sum(v["p"] for v in per.values()) / n,
            "macro_r": sum(v["r"] for v in per.values()) / n,
            "micro_f1": mf, "micro_p": mp, "micro_r": mr, "per_label": per}

def error_analysis(gold, pred, texts=None, n_examples=5):
    cats = Counter()
    confusion = Counter()
    boundary_kind = Counter()
    by_label = defaultdict(Counter)
    examples = defaultdict(list)
    for did, g in gold.items():
        g = set(map(tuple, g)); p = set(map(tuple, pred.get(did, [])))
        P = sorted(p)
        for x in sorted(g - p):
            s, e, l = x
            ov = [y for y in P if y[0] < e and y[1] > s]
            same_span = [y for y in ov if y[0] == s and y[1] == e]
            if same_span:
                c = "type"; confusion[(l, same_span[0][2])] += 1
            elif any(y[2] == l for y in ov):
                c = "boundary"
                y = next(y for y in ov if y[2] == l)
                boundary_kind[("start " + ("early" if y[0] < s else "late" if y[0] > s else "ok")) + " / " +
                              ("end " + ("early" if y[1] < e else "late" if y[1] > e else "ok"))] += 1
            elif ov:
                c = "both"
            else:
                c = "missed"
            cats[c] += 1; by_label[l][c] += 1
            if texts is not None and len(examples[c]) < n_examples:
                t = texts[did]
                pr = "; ".join(f"[{t[y[0]:y[1]]}]{y[2]}" for y in ov[:2])
                examples[c].append(f"{l}: [{t[s:e]}] | pred: {pr or '-'}")
        G = sorted(g)
        for y in sorted(p - g):
            if not any(x[0] < y[1] and x[1] > y[0] for x in G):
                cats["spurious"] += 1; by_label[y[2]]["spurious"] += 1
                if texts is not None and len(examples["spurious"]) < n_examples:
                    t = texts[did]
                    examples["spurious"].append(f"{y[2]}: [{t[y[0]:y[1]]}]")
    return {"categories": dict(cats), "type_confusion": {f"{a}->{b}": n for (a, b), n in confusion.most_common(15)},
            "boundary_kind": dict(boundary_kind), "by_label": {k: dict(v) for k, v in by_label.items()},
            "examples": dict(examples)}

def format_table(scores):
    lines = [f"{'label':10s} {'P':>6s} {'R':>6s} {'F1':>6s} {'support':>8s}"]
    for l, v in scores["per_label"].items():
        lines.append(f"{l:10s} {v['p']:6.3f} {v['r']:6.3f} {v['f1']:6.3f} {v['support']:8d}")
    lines.append(f"{'MACRO':10s} {scores['macro_p']:6.3f} {scores['macro_r']:6.3f} {scores['macro_f1']:6.3f}")
    lines.append(f"{'micro':10s} {scores['micro_p']:6.3f} {scores['micro_r']:6.3f} {scores['micro_f1']:6.3f}")
    return "\n".join(lines)

In [ ]:
%%writefile {WORK}/p2lner/backbones.py
"""Load a Phase-1 `model_final.pt` and expose it as a token encoder: ids [B,T] -> hidden states [B,T,C].

All four architectures share the block layout  x += attn(norm1(x)); x += ffn(norm2(x)), so one forward loop
serves all of them. Attention is recomputed here from each block's own projection layers so that the mask
can be chosen per experiment:

    attention="causal"         token i sees tokens <= i            (how A and D were pretrained)
    attention="bidirectional"  token i sees every real token       (how B's encoder / C's prefix were pretrained)
    D additionally keeps its sliding window: |i - j| < window (bidirectional) or 0 <= i - j < window (causal).

Padding is on the right and masked out as keys. The LM head is dropped (NER uses its own head).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from .phase1_models import MODEL_CLASSES

ARCH_OF_CLASS = {"LlamaLM": "A", "EncDecLM": "B", "HybridLM": "C", "EfficientLM": "D"}

def _clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        for p in ("_orig_mod.", "module."):
            while k.startswith(p):
                k = k[len(p):]
        out[k] = v
    return out

def attention_forward(attn, x, rope, mask):
    B, T, _ = x.shape
    H, hd = attn.n_head, attn.hd
    if hasattr(attn, "qkv"):
        q, k, v = attn.qkv(x).view(B, T, 3, H, hd).permute(2, 0, 3, 1, 4)
    else:
        q = attn.wq(x).view(B, T, H, hd).transpose(1, 2)
        k, v = attn.wkv(x).view(B, T, 2, attn.n_kv, hd).permute(2, 0, 3, 1, 4)
        if attn.n_kv != H:
            rep = H // attn.n_kv
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
    q, k = rope(q, 0), rope(k, 0)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                       dropout_p=attn.dropout if attn.training else 0.0)
    return attn.resid_drop(attn.proj(y.transpose(1, 2).reshape(B, T, -1)))

class NERBackbone(nn.Module):
    def __init__(self, arch, tok_emb, blocks, norm, rope, max_len, attention="bidirectional", window=None,
                 seg_emb=None, seg_id=0, grad_ckpt=False):
        super().__init__()
        assert attention in ("causal", "bidirectional")
        self.arch, self.attention, self.window, self.max_len = arch, attention, window, max_len
        self.tok_emb, self.blocks, self.norm, self.rope = tok_emb, blocks, norm, rope
        self.seg_emb, self.seg_id = seg_emb, seg_id
        self.d_model = tok_emb.weight.size(1)
        self.grad_ckpt = grad_ckpt
        self.soft_prompt = None

    def build_mask(self, keep):
        T = keep.size(1)
        i = torch.arange(T, device=keep.device)
        rel = i[:, None] - i[None, :]
        allow = torch.ones(T, T, dtype=torch.bool, device=keep.device)
        if self.attention == "causal":
            allow &= rel >= 0
        if self.window is not None:
            allow &= rel.abs() < self.window
        return (allow[None] & keep[:, None, :])[:, None]

    def forward(self, ids, keep):
        x = self.tok_emb(ids)
        if self.seg_emb is not None:
            x = x + self.seg_emb.weight[self.seg_id]
        n_p = 0
        if self.soft_prompt is not None:
            n_p = self.soft_prompt.size(0)
            x = torch.cat([self.soft_prompt[None].expand(x.size(0), -1, -1).to(x.dtype), x], 1)
            keep = torch.cat([torch.ones(keep.size(0), n_p, dtype=torch.bool, device=keep.device), keep], 1)
        assert x.size(1) <= self.max_len, f"sequence {x.size(1)} > positional table {self.max_len}"
        mask = self.build_mask(keep)
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = checkpoint(self._block, blk, x, mask, use_reentrant=False)
            else:
                x = self._block(blk, x, mask)
        x = self.norm(x)
        return x[:, n_p:]

    def _block(self, blk, x, mask):
        x = x + attention_forward(blk.attn, blk.norm1(x), self.rope, mask)
        return x + blk.ffn(blk.norm2(x))

def load_phase1(ckpt_path):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cls_name = ck.get("model_class")
    kw = dict(ck["model_kw"])
    if cls_name is None:
        cls_name = ("EncDecLM" if "enc_block" in kw else "HybridLM" if "n_segments" in kw
                    else "EfficientLM" if "window" in kw else "LlamaLM")
    model = MODEL_CLASSES[cls_name](**kw)
    missing, unexpected = model.load_state_dict(_clean_state_dict(ck["model"]), strict=False)
    missing = [m for m in missing if not m.endswith("lm_head.weight")]
    assert not missing and not unexpected, f"checkpoint mismatch: missing={missing} unexpected={unexpected}"
    return model, cls_name, kw

def load_backbone(ckpt_path, attention="bidirectional", keep_layers=None, c_segment=0, grad_ckpt=False):
    model, cls_name, kw = load_phase1(ckpt_path)
    arch = ARCH_OF_CLASS[cls_name]
    if arch == "B":
        blocks, norm, rope, max_len, window, seg = model.enc_blocks, model.enc_norm, model.enc_rope, kw["enc_block"], None, None
    elif arch == "D":
        blocks, norm, rope, max_len, window, seg = model.blocks, model.norm_f, model.rope, model.max_pos, model.window, None
    else:
        blocks, norm, rope, max_len, window = model.blocks, model.norm_f, model.rope, kw["block_size"], None
        seg = model.seg_emb if arch == "C" else None
    n_total_layers = len(blocks)
    if keep_layers:
        blocks = nn.ModuleList(list(blocks)[:keep_layers])
    bb = NERBackbone(arch, model.tok_emb, blocks, norm, rope, max_len, attention, window, seg, c_segment, grad_ckpt)
    del model
    info = {"arch": arch, "class": cls_name, "model_kw": {k: v for k, v in kw.items()}, "phase1_layers": n_total_layers,
            "layers_used": len(blocks), "d_model": bb.d_model, "max_len": max_len, "window": window,
            "attention": attention, "params": sum(p.numel() for p in bb.parameters())}
    return bb, info

In [ ]:
%%writefile {WORK}/p2lner/peft.py
"""Fine-tuning / parameter-efficient adaptation methods, applied in place to an NERBackbone.

    full     every backbone parameter trainable
    partial  only the last `train_last_k` blocks (+ final norm) trainable
    frozen   backbone frozen, only the head trains (linear probe: measures raw Phase-1 transfer)
    lora     W x + (alpha/r) B A x                           (Hu et al., 2021)
    lorma    LoRMA+ : (I + (alpha/r) B A) W x                 (Bihany, Patel & Modi, 2025)
             LoRMApi: R_pi(B A) W x, rows of BA cyclically rotated (row i by i) -> full rank
    adapter  Houlsby bottleneck after the attention and FFN output projections: y + U act(D y)
    bitfit   zero-initialised bias added to every block linear + RMSNorm gains (the Phase-1 linears have no
             biases, so BitFit's "train only the biases" becomes "add and train only biases")
    prompt   n learned vectors prepended to the input embeddings (Lester et al., 2021)

The NER head is always trainable. New PEFT parameters are named lora_*, lorma_*, adapter_*, bitfit_* or
soft_prompt so the trainer can give them their own learning rate.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

NEW_PARAM_TAGS = ("lora_", "lorma_", "adapter_", "bitfit_", "soft_prompt")

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B)

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, variant="plus"):
        super().__init__()
        self.base, self.variant, self.scale = base, variant, alpha / r
        d = base.out_features
        self.d = d
        self.lorma_A = nn.Parameter(torch.empty(r, d))
        self.lorma_B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
            nn.init.zeros_(self.lorma_B)
        elif variant == "pi":
            nn.init.zeros_(self.lorma_A); self.lorma_A.data[0, 0] = 1.0
            nn.init.normal_(self.lorma_B, std=0.02); self.lorma_B.data[:, 0] = 1.0
        else:
            raise ValueError(variant)

    def rotated(self):
        M = self.lorma_B @ self.lorma_A
        d = self.d
        M2 = torch.cat([M, M], 1).contiguous()
        return M2.as_strided((d, d), (2 * d - 1, 1), storage_offset=d)

    def forward(self, x):
        h = self.base(x)
        if self.variant == "plus":
            return h + self.scale * F.linear(F.linear(h, self.lorma_A), self.lorma_B)
        return F.linear(h, self.rotated().to(h.dtype))

class HoulsbyAdapter(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        d = base.out_features
        self.adapter_down = nn.Linear(d, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.adapter_down.weight, std=0.02); nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight); nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.adapter_up(F.gelu(self.adapter_down(y)))

class BitFitLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bitfit_bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bitfit_bias

def _block_linears(blk, which=("attn", "ffn")):
    out = []
    if "attn" in which:
        for a in ("wq", "wkv", "qkv", "proj"):
            if hasattr(blk.attn, a):
                out.append((blk.attn, a, "attn_out" if a == "proj" else "attn_in"))
    if "ffn" in which:
        out.append((blk.ffn, "w12", "ffn_in"))
        out.append((blk.ffn, "proj", "ffn_out"))
    return out

def apply_method(bb, cfg):
    name = cfg["name"]
    for p in bb.parameters():
        p.requires_grad = name == "full"
    if name in ("full", "frozen"):
        return bb
    if name == "partial":
        k = cfg.get("train_last_k", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in bb.norm.parameters():
            p.requires_grad = True
        return bb
    if name == "prompt":
        bb.soft_prompt = nn.Parameter(bb.tok_emb.weight[torch.randint(0, 20000, (cfg.get("n_prompt", 20),))].detach().clone())
        return bb
    targets = cfg.get("targets", ["attn", "ffn"])
    for blk in bb.blocks:
        if name in ("lora", "lorma"):
            for parent, attr, kind in _block_linears(blk, targets):
                base = getattr(parent, attr)
                if name == "lora":
                    setattr(parent, attr, LoRALinear(base, cfg.get("r", 8), cfg.get("alpha", 16), cfg.get("dropout", 0.05)))
                else:
                    if cfg.get("variant", "plus") == "pi" and base.out_features > cfg.get("pi_max_dim", 2048):
                        continue
                    setattr(parent, attr, LoRMALinear(base, cfg.get("r", 8), cfg.get("alpha", 16), cfg.get("variant", "plus")))
        elif name == "adapter":
            for parent, attr, kind in _block_linears(blk):
                if kind in ("attn_out", "ffn_out"):
                    setattr(parent, attr, HoulsbyAdapter(getattr(parent, attr), cfg.get("bottleneck", 64)))
        elif name == "bitfit":
            for parent, attr, kind in _block_linears(blk):
                setattr(parent, attr, BitFitLinear(getattr(parent, attr)))
            for p in (blk.norm1.weight, blk.norm2.weight):
                p.requires_grad = True
        else:
            raise ValueError(f"unknown method {name}")
    if name == "bitfit":
        bb.norm.weight.requires_grad = True
    if cfg.get("train_norms", False):
        for blk in bb.blocks:
            blk.norm1.weight.requires_grad = blk.norm2.weight.requires_grad = True
        bb.norm.weight.requires_grad = True
    return bb

def is_new_param(name):
    return any(t in name for t in NEW_PARAM_TAGS)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": train, "trainable_pct": round(100 * train / total, 3)}

In [ ]:
%%writefile {WORK}/p2lner/heads.py
"""Prediction heads on top of the backbone's hidden states.

linear : dropout + Linear(d_model -> n_tags); token-level softmax, decoded with BIO-constrained Viterbi.
crf    : same emissions + a linear-chain CRF (learned tag-transition scores, BIO-illegal transitions fixed
         to -1e4). Trained with the sequence negative log-likelihood, decoded with Viterbi.
"""
import torch
import torch.nn as nn

from .spans import bio_constraints

NEG = -1e4

class LinearHead(nn.Module):
    def __init__(self, d_model, n_tags, dropout=0.1, hidden=0):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        if hidden:
            self.proj = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_tags))
        else:
            self.proj = nn.Linear(d_model, n_tags)

    def forward(self, h):
        return self.proj(self.drop(h))

class CRF(nn.Module):
    def __init__(self, tagset):
        super().__init__()
        K = tagset.n
        self.K = K
        self.trans = nn.Parameter(torch.zeros(K, K))
        self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        s, t = bio_constraints(tagset)
        self.register_buffer("start_mask", torch.tensor(s).clamp(min=NEG))
        self.register_buffer("trans_mask", torch.tensor(t).clamp(min=NEG))

    def full_trans(self):
        return self.trans + self.trans_mask, self.start + self.start_mask, self.end

    def nll(self, emis, tags, mask):
        trans, _, end = self.full_trans()
        start = self.start
        B, T, K = emis.shape
        tags = tags.clamp(min=0)
        score = start[tags[:, 0]] + emis[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, T):
            m = mask[:, t]
            s_t = trans[tags[:, t - 1], tags[:, t]] + emis[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            score = score + s_t * m
        lengths = mask.sum(1)
        last = tags.gather(1, (lengths - 1).clamp(min=0)[:, None]).squeeze(1)
        score = score + end[last]
        alpha = start[None] + emis[:, 0]
        for t in range(1, T):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + emis[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logZ = torch.logsumexp(alpha + end[None], dim=1)
        return ((logZ - score) / lengths.clamp(min=1)).mean()

In [ ]:
%%writefile {WORK}/p2lner/model.py
"""NERModel = Phase-1 backbone (+ PEFT) + head, with the training losses.

Losses (linear head):
    CE      token cross-entropy over BIO tags (optional class weights / label smoothing)
    KD      alpha * CE + (1 - alpha) * T^2 * KL( softmax(teacher / T) || softmax(student / T) )   (Hinton et al., 2015)
CRF head: sequence NLL (KD is only supported with the linear head).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

from .backbones import load_backbone
from .heads import CRF, LinearHead
from .peft import apply_method

class NERModel(nn.Module):
    def __init__(self, backbone, tagset, head="linear", dropout=0.1, head_hidden=0):
        super().__init__()
        self.backbone, self.tagset, self.head_type = backbone, tagset, head
        self.head = LinearHead(backbone.d_model, tagset.n, dropout, head_hidden)
        self.crf = CRF(tagset) if head == "crf" else None
        self.register_buffer("class_w", torch.ones(tagset.n))

    def emissions(self, ids, keep):
        return self.head(self.backbone(ids, keep)).float()

    def loss(self, emis, labels, keep, cfg, teacher=None):
        if self.crf is not None:
            return self.crf.nll(emis, labels, keep)
        K = emis.size(-1)
        ce = F.cross_entropy(emis.view(-1, K), labels.view(-1), weight=self.class_w, ignore_index=-100,
                             label_smoothing=cfg.get("label_smoothing", 0.0))
        if cfg.get("focal_gamma", 0):
            lp = F.log_softmax(emis, -1).view(-1, K)
            y = labels.view(-1); ok = y != -100
            lpt = lp[ok].gather(1, y[ok, None]).squeeze(1)
            ce = (-(1 - lpt.exp()) ** cfg["focal_gamma"] * lpt * self.class_w[y[ok]]).mean()
        if teacher is None:
            return ce
        T, a = cfg.get("kd_T", 2.0), cfg.get("kd_alpha", 0.5)
        ok = (labels != -100)
        s = F.log_softmax(emis[ok] / T, -1)
        t = F.log_softmax(teacher[ok].float() / T, -1)
        kl = F.kl_div(s, t, log_target=True, reduction="batchmean")
        return a * ce + (1 - a) * T * T * kl

def build_model(cfg, tagset):
    bcfg = cfg["backbone"]
    bb, info = load_backbone(bcfg["ckpt"], attention=bcfg.get("attention", "bidirectional"),
                             keep_layers=bcfg.get("keep_layers"), c_segment=bcfg.get("c_segment", 0),
                             grad_ckpt=bcfg.get("grad_ckpt", False))
    apply_method(bb, cfg["method"])
    m = NERModel(bb, tagset, cfg["head"]["type"], cfg["head"].get("dropout", 0.1), cfg["head"].get("hidden", 0))
    return m, info

def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items()
            if k in keep or k.startswith("head.") or k.startswith("crf.") or k == "class_w"}

In [ ]:
%%writefile {WORK}/p2lner/train.py
"""Training, sliding-window inference, evaluation and run management.

Long documents are cut into windows of `max_len` tokens with stride `stride` (overlap = max_len - stride).
At inference every token gets a weighted average of its scores over all windows that contain it:
    bidirectional: weight = distance to the nearer window edge (centre tokens have context on both sides)
    causal       : weight = position inside the window (later tokens have seen more left context)
"""
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from . import data as D
from .metrics import error_analysis, format_table, strict_scores
from .model import build_model, trainable_state
from .peft import count_params, is_new_param
from .spans import TagSet, decode_doc, learn_trim_rules, reachability, spans_to_tags, trim_spans
from .tokenizer import BPETokenizer

DEFAULTS = {
    "run_name": "debug",
    "out_dir": "runs",
    "data_dir": "",
    "backbone": {"ckpt": "", "attention": "bidirectional", "keep_layers": None, "c_segment": 0, "grad_ckpt": False},
    "tokenizer": {"path": "", "fine_punct": True},
    "window": {"max_len": 512, "stride": 384, "random_offset": True},
    "method": {"name": "full"},
    "head": {"type": "linear", "dropout": 0.1, "hidden": 0},
    "loss": {"class_weight": "none", "cw_power": 0.5, "cw_max": 10.0, "label_smoothing": 0.0, "focal_gamma": 0},
    "kd": {"teacher_run": "", "kd_alpha": 0.5, "kd_T": 2.0},
    "decode": {"strategy": "bio_viterbi", "trim": True},
    "split": {"mode": "random", "dev_frac": 0.15, "seed": 1337},
    "train": {"epochs": 10, "batch_size": 8, "grad_accum": 1, "lr_backbone": 5e-5, "lr_new": 1e-3,
              "weight_decay": 0.01, "warmup_frac": 0.1, "patience": 3, "grad_clip": 1.0, "seed": 1337,
              "eval_every_epochs": 1, "num_workers": 0},
    "predict_test": True,
    "ckpt_root": "",
}

def merge(base, over):
    out = json.loads(json.dumps(base))
    for k, v in (over or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = merge(out[k], v)
        else:
            out[k] = v
    return out

def set_by_path(cfg, dotted, value):
    try:
        value = json.loads(value)
    except (json.JSONDecodeError, TypeError):
        pass
    d = cfg
    ks = dotted.split(".")
    for k in ks[:-1]:
        d = d.setdefault(k, {})
    d[ks[-1]] = value

CKPT_FOLDERS = {"A": "A_llama_decoder", "A2": "A_llama_decoder_v2", "B": "B_encoder_decoder",
                "C": "C_hybrid_prefix_mlm", "D": "D_efficient_swa_mqa"}

def resolve_ckpt(spec, roots=()):
    if not spec.startswith("@"):
        return spec
    folder = CKPT_FOLDERS[spec[1:]]
    for r in [*roots, "/kaggle/input", "/kaggle/working"]:
        r = Path(r)
        if not r.exists():
            continue
        direct = r / folder / "model_final.pt"
        if direct.exists():
            return str(direct)
        for p in r.rglob("model_final.pt"):
            if p.parent.name == folder:
                return str(p)
    raise FileNotFoundError(f"{folder}/model_final.pt not found under {roots} or /kaggle/input")

def resolve_paths(cfg):
    roots = [cfg["ckpt_root"]] if cfg.get("ckpt_root") else []
    cfg["backbone"]["ckpt"] = resolve_ckpt(cfg["backbone"]["ckpt"], roots)
    return cfg

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def get_tokenizer(cfg):
    p = cfg["tokenizer"].get("path") or str(Path(cfg["backbone"]["ckpt"]).parent / "tokenizer.json")
    return BPETokenizer.load(p, fine_punct=cfg["tokenizer"].get("fine_punct", True))

def prepare(docs, tok, tagset):
    out = {}
    for did, d in docs.items():
        enc = tok.encode_with_offsets(d["text"])
        item = {"enc": enc, "ids": np.asarray(enc.ids, dtype=np.int64)}
        if d.get("gold") is not None:
            item["tags"], _ = spans_to_tags(d["text"], enc, d["gold"], tagset)
        out[did] = item
    return out

def windows(n, L, stride, offset=0):
    if n <= L:
        return [(0, n)]
    starts = list(range(-offset, n - L, stride)) if offset else list(range(0, n - L, stride))
    starts = [max(0, s) for s in starts]
    starts.append(n - L)
    return sorted(set((s, s + L) for s in starts))

def collate(items, prep, pad_id, teacher=None, K=None):
    T = max(e - s for _, s, e in items)
    B = len(items)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    keep = torch.zeros((B, T), dtype=torch.bool)
    labels = torch.full((B, T), -100, dtype=torch.long)
    tch = torch.zeros((B, T, K), dtype=torch.float16) if teacher is not None else None
    for b, (did, s, e) in enumerate(items):
        n = e - s
        ids[b, :n] = torch.from_numpy(prep[did]["ids"][s:e])
        keep[b, :n] = True
        if "tags" in prep[did]:
            labels[b, :n] = torch.from_numpy(prep[did]["tags"][s:e])
        if tch is not None:
            tch[b, :n] = torch.from_numpy(teacher[did][s:e])
    return ids, keep, labels, tch

def amp_dtype(device):
    if device != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

@torch.no_grad()
def predict_scores(model, prep, L, stride, batch_size=16, device="cuda"):
    model.eval()
    dt = amp_dtype(device)
    causal = model.backbone.attention == "causal"
    K = model.tagset.n
    acc = {d: np.zeros((len(p["ids"]), K), dtype=np.float32) for d, p in prep.items()}
    wsum = {d: np.zeros(len(p["ids"]), dtype=np.float32) for d, p in prep.items()}
    items = [(d, s, e) for d, p in prep.items() for s, e in windows(len(p["ids"]), L, stride)]
    for i in range(0, len(items), batch_size):
        chunk = items[i:i + batch_size]
        ids, keep, _, _ = collate(chunk, prep, pad_id=0)
        ids, keep = ids.to(device), keep.to(device)
        with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
            em = model.emissions(ids, keep)
        sc = (em if model.crf is not None else F.log_softmax(em, -1)).float().cpu().numpy()
        for b, (d, s, e) in enumerate(chunk):
            n = e - s
            pos = np.arange(n, dtype=np.float32)
            w = (pos + 1) if causal else np.minimum(pos + 1, n - pos)
            acc[d][s:e] += sc[b, :n] * w[:, None]
            wsum[d][s:e] += w
    return {d: acc[d] / np.maximum(wsum[d], 1e-6)[:, None] for d in acc}

def decode_all(model, docs, prep, scores, cfg, trim_rules):
    trans = start = end = None
    if model.crf is not None:
        tr, st, en = model.crf.full_trans()
        trans, start, end = (x.detach().float().cpu().numpy() for x in (tr, st, en))
    out = {}
    for did, d in docs.items():
        sp = decode_doc(d["text"], prep[did]["enc"], scores[did], model.tagset, cfg["decode"]["strategy"],
                        trans, start, end)
        if cfg["decode"].get("trim", True):
            sp = trim_spans(d["text"], sp, trim_rules)
        out[did] = sorted(set(sp))
    return out

def evaluate(model, docs, prep, cfg, trim_rules, device):
    sc = predict_scores(model, prep, cfg["window"]["max_len"], cfg["window"]["stride"],
                        cfg["train"]["batch_size"] * 2, device)
    pred = decode_all(model, docs, prep, sc, cfg, trim_rules)
    gold = {k: d["gold"] for k, d in docs.items()}
    return strict_scores(gold, pred, model.tagset.labels), pred

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(obj, open(path, "w", encoding="utf-8"), indent=2, default=str)

def load_run(run_dir, device="cuda"):
    run_dir = Path(run_dir)
    cfg = json.load(open(run_dir / "config.json"))
    labels = json.load(open(run_dir / "labels.json"))
    tagset = TagSet(labels)
    model, _ = build_model(cfg, tagset)
    sd = torch.load(run_dir / "best.pt", map_location="cpu")
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, unexpected
    trim_rules = json.load(open(run_dir / "trim_rules.json"))
    return model.to(device), cfg, tagset, trim_rules

def write_submission(pred, test_ids, labels, out_dir):
    import zipfile
    assert set(pred) == set(test_ids), "prediction IDs must match the official test IDs exactly"
    lab = set(labels)
    obj = {}
    for d in test_ids:
        rows = []
        for s, e, l in pred[d]:
            assert l in lab and 0 <= s < e, (d, s, e, l)
            rows.append([int(s), int(e), l])
        obj[d] = rows
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pj = out_dir / "prediction.json"
    json.dump(obj, open(pj, "w", encoding="utf-8"))
    zp = out_dir / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp

def run(cfg, data=None, device=None, log=print):
    cfg = resolve_paths(merge(DEFAULTS, cfg))
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    seed_all(cfg["train"]["seed"])
    run_dir = Path(cfg["out_dir"]) / cfg["run_name"]
    run_dir.mkdir(parents=True, exist_ok=True)
    save_json(cfg, run_dir / "config.json")

    data = data or D.load_phase2(cfg["data_dir"])
    labels = data["labels"]
    tagset = TagSet(labels)
    save_json(labels, run_dir / "labels.json")
    train_docs, dev_docs = D.make_split(data, **cfg["split"])
    log(f"[data] train docs {len(train_docs)} | dev docs {len(dev_docs)} | test docs {len(data['test'])}")
    tok = get_tokenizer(cfg)
    trim_rules = learn_trim_rules(train_docs) if cfg["decode"].get("trim", True) else None
    save_json(trim_rules, run_dir / "trim_rules.json")
    reach, _, skipped = reachability(train_docs, tok, tagset)
    log(f"[data] BIO round-trip ceiling on train: macro-F1 {reach['macro_f1']:.4f} (overlaps dropped: {skipped})")
    t0 = time.time()
    prep_tr, prep_dev = prepare(train_docs, tok, tagset), prepare(dev_docs, tok, tagset)
    log(f"[data] tokenised in {time.time() - t0:.0f}s | train tokens {sum(len(p['ids']) for p in prep_tr.values()):,}")

    teacher = None
    if cfg["kd"].get("teacher_run"):
        assert cfg["head"]["type"] == "linear", "KD is implemented for the linear head"
        tm, tcfg, ttags, _ = load_run(cfg["kd"]["teacher_run"], device)
        assert ttags.tags == tagset.tags and tcfg["tokenizer"].get("fine_punct", True) == cfg["tokenizer"].get("fine_punct", True)
        teacher = {d: s.astype(np.float16) for d, s in predict_scores(
            tm, prep_tr, tcfg["window"]["max_len"], tcfg["window"]["stride"], 16, device).items()}
        t_eval, _ = evaluate(tm, dev_docs, prep_dev, tcfg, trim_rules, device) if dev_docs else ({"macro_f1": None}, None)
        log(f"[kd] teacher {cfg['kd']['teacher_run']} | dev macro-F1 {t_eval['macro_f1']}")
        save_json({"teacher_run": cfg["kd"]["teacher_run"], "teacher_dev_macro_f1": t_eval["macro_f1"],
                   "teacher_params": count_params(tm)}, run_dir / "teacher.json")
        del tm
        torch.cuda.empty_cache() if device == "cuda" else None

    model, info = build_model(cfg, tagset)
    n_prompt = 0 if model.backbone.soft_prompt is None else model.backbone.soft_prompt.size(0)
    L, stride = cfg["window"]["max_len"], cfg["window"]["stride"]
    assert L + n_prompt <= info["max_len"], f"max_len {L} (+{n_prompt} prompt) > backbone positions {info['max_len']}"
    if cfg["loss"]["class_weight"] == "sqrt_inv":
        cnt = np.bincount(np.concatenate([p["tags"] for p in prep_tr.values()]), minlength=tagset.n).astype(np.float64)
        ref = np.median(cnt[cnt > 0])
        w = np.clip((ref / np.maximum(cnt, 1)) ** cfg["loss"]["cw_power"], 1.0 / cfg["loss"]["cw_max"], cfg["loss"]["cw_max"])
        model.class_w.copy_(torch.tensor(w / w.mean(), dtype=torch.float32))
    model.to(device)
    pc = count_params(model)
    info.update(params=pc)
    save_json(info, run_dir / "model_info.json")
    log(f"[model] arch {info['arch']} ({info['class']}) | layers {info['layers_used']}/{info['phase1_layers']} | "
        f"attention {info['attention']} | method {cfg['method']['name']} | head {cfg['head']['type']}")
    log(f"[model] params total {pc['total'] / 1e6:.2f}M | trainable {pc['trainable'] / 1e6:.3f}M ({pc['trainable_pct']}%)")

    tc = cfg["train"]
    groups = {"bb_decay": [], "bb_nodecay": [], "new_decay": [], "new_nodecay": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        new = is_new_param(n) or n.startswith("head.") or n.startswith("crf.")
        groups[("new" if new else "bb") + ("_decay" if p.dim() >= 2 else "_nodecay")].append(p)
    opt = torch.optim.AdamW([
        {"params": groups["bb_decay"], "lr": tc["lr_backbone"], "weight_decay": tc["weight_decay"]},
        {"params": groups["bb_nodecay"], "lr": tc["lr_backbone"], "weight_decay": 0.0},
        {"params": groups["new_decay"], "lr": tc["lr_new"], "weight_decay": tc["weight_decay"]},
        {"params": groups["new_nodecay"], "lr": tc["lr_new"], "weight_decay": 0.0}])
    base_lrs = [g["lr"] for g in opt.param_groups]
    n_win = sum(len(windows(len(p["ids"]), L, stride)) for p in prep_tr.values())
    steps_per_epoch = math.ceil(n_win / (tc["batch_size"] * tc["grad_accum"]))
    total = steps_per_epoch * tc["epochs"]
    warm = max(1, int(tc["warmup_frac"] * total))
    dt = amp_dtype(device)
    scaler = torch.amp.GradScaler("cuda", enabled=(dt == torch.float16))
    log(f"[train] {n_win} windows/epoch | {steps_per_epoch} steps/epoch | {total} steps | amp {dt}")
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    hist, best, bad_epochs, step = [], -1.0, 0, 0
    rng = random.Random(tc["seed"])
    t_train = time.time()
    for ep in range(1, tc["epochs"] + 1):
        model.train()
        items = []
        for d, p in prep_tr.items():
            off = rng.randrange(stride) if cfg["window"].get("random_offset") else 0
            items += [(d, s, e) for s, e in windows(len(p["ids"]), L, stride, off)]
        rng.shuffle(items)
        tot_loss, nb = 0.0, 0
        for bi in range(0, len(items), tc["batch_size"]):
            ids, keep, lab, tch = collate(items[bi:bi + tc["batch_size"]], prep_tr, tok.pad_id, teacher, tagset.n)
            ids, keep, lab = ids.to(device), keep.to(device), lab.to(device)
            tch = tch.to(device) if tch is not None else None
            with torch.autocast("cuda", dtype=dt, enabled=dt is not None):
                em = model.emissions(ids, keep)
            loss = model.loss(em, lab, keep, {**cfg["loss"], **cfg["kd"]}, tch) / tc["grad_accum"]
            scaler.scale(loss).backward()
            tot_loss += loss.item() * tc["grad_accum"]; nb += 1
            if nb % tc["grad_accum"] == 0:
                lr_mult = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))
                for g, b in zip(opt.param_groups, base_lrs):
                    g["lr"] = b * lr_mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], tc["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
        rec = {"epoch": ep, "train_loss": tot_loss / max(nb, 1), "minutes": round((time.time() - t_train) / 60, 2)}
        if dev_docs and ep % tc["eval_every_epochs"] == 0:
            sc, _ = evaluate(model, dev_docs, prep_dev, cfg, trim_rules, device)
            rec.update(dev_macro_f1=sc["macro_f1"], dev_macro_p=sc["macro_p"], dev_macro_r=sc["macro_r"], dev_micro_f1=sc["micro_f1"])
            if sc["macro_f1"] > best:
                best, bad_epochs = sc["macro_f1"], 0
                torch.save(trainable_state(model), run_dir / "best.pt"); rec["saved"] = True
            else:
                bad_epochs += 1
        hist.append(rec)
        log("[epoch] " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        save_json(hist, run_dir / "history.json")
        if dev_docs and bad_epochs >= tc["patience"]:
            log(f"[train] early stop: no dev improvement for {tc['patience']} epochs")
            break
    train_min = (time.time() - t_train) / 60
    if not dev_docs:
        torch.save(trainable_state(model), run_dir / "best.pt")
    peak_mem = torch.cuda.max_memory_allocated() / 2**30 if device == "cuda" else None

    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device), strict=False)
    summary = {"run": cfg["run_name"], "arch": info["arch"], "method": cfg["method"], "head": cfg["head"]["type"],
               "attention": info["attention"], "params_total": pc["total"], "params_trainable": pc["trainable"],
               "trainable_pct": pc["trainable_pct"], "train_minutes": round(train_min, 1),
               "peak_train_mem_gb": peak_mem, "best_dev_macro_f1": best if dev_docs else None,
               "bio_ceiling_macro_f1": reach["macro_f1"]}
    if dev_docs:
        t0 = time.time()
        sc, pred = evaluate(model, dev_docs, prep_dev, cfg, trim_rules, device)
        dt_inf = time.time() - t0
        texts = {k: d["text"] for k, d in dev_docs.items()}
        err = error_analysis({k: d["gold"] for k, d in dev_docs.items()}, pred, texts)
        save_json(sc, run_dir / "dev_scores.json"); save_json(err, run_dir / "dev_errors.json")
        save_json({k: [list(x) for x in v] for k, v in pred.items()}, run_dir / "dev_prediction.json")
        ntok = sum(len(p["ids"]) for p in prep_dev.values())
        summary.update(dev=dict((k, sc[k]) for k in ("macro_f1", "macro_p", "macro_r", "micro_f1")),
                       infer_tokens_per_sec=round(ntok / dt_inf), error_categories=err["categories"])
        log("\n" + format_table(sc))
        log(f"[errors] {err['categories']}")
    if cfg.get("predict_test", True):
        t_prep = prepare({k: {"text": v} for k, v in data["test"].items()}, tok, tagset)
        tsc = predict_scores(model, t_prep, L, stride, tc["batch_size"] * 2, device)
        tpred = decode_all(model, {k: {"text": v} for k, v in data["test"].items()}, t_prep, tsc, cfg, trim_rules)
        zp = write_submission(tpred, list(data["test"]), labels, run_dir / "submission")
        summary["submission"] = str(zp)
        summary["test_entities"] = sum(len(v) for v in tpred.values())
        log(f"[submit] {zp} | {summary['test_entities']} entities over {len(tpred)} docs")
    save_json(summary, run_dir / "summary.json")
    return summary, model

In [ ]:
%%writefile {WORK}/p2lner/run.py
"""CLI.

    python -m p2lner.run --config configs/E1_C_full.json --set data_dir=/kaggle/working/p2data/Phase_2_LNER \
                         --set backbone.ckpt=/kaggle/input/.../model_final.pt
    python -m p2lner.run --eda --config configs/E1_C_full.json      # data statistics + BIO reachability only
    python -m p2lner.run --predict runs/E1_C_full                    # re-predict test from a saved run
"""
import argparse
import json
from pathlib import Path

from . import data as D
from .spans import TagSet, reachability
from .train import DEFAULTS, decode_all, get_tokenizer, load_run, merge, predict_scores, prepare, resolve_paths, run, set_by_path, write_submission

def eda(cfg):
    data = D.load_phase2(cfg["data_dir"])
    tagset = TagSet(data["labels"])
    train_docs, _ = D.make_split(data, mode="none")
    print("labels:", data["labels"])
    print("gold checks:", D.validate_gold(train_docs, data["labels"]))
    for fine in (False, True):
        cfg["tokenizer"]["fine_punct"] = fine
        tok = get_tokenizer(cfg)
        sc, bad, skipped = reachability(train_docs, tok, tagset)
        print(f"\n== tokenisation fine_punct={fine}: BIO round-trip macro-F1 {sc['macro_f1']:.4f} | micro {sc['micro_f1']:.4f} "
              f"| unreachable {len(bad)} | overlaps dropped {skipped}")
        for b in bad[:10]:
            print("   ", b)
    print(json.dumps(D.eda(train_docs, data["labels"], tok), indent=1))
    print("test docs:", len(data["test"]))

def repredict(run_dir, data_dir=None, device="cuda"):
    model, cfg, tagset, trim_rules = load_run(run_dir, device)
    data = D.load_phase2(data_dir or cfg["data_dir"])
    tok = get_tokenizer(cfg)
    docs = {k: {"text": v} for k, v in data["test"].items()}
    prep = prepare(docs, tok, tagset)
    sc = predict_scores(model, prep, cfg["window"]["max_len"], cfg["window"]["stride"], 16, device)
    pred = decode_all(model, docs, prep, sc, cfg, trim_rules)
    return write_submission(pred, list(data["test"]), data["labels"], Path(run_dir) / "submission")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="")
    ap.add_argument("--set", action="append", default=[], help="dotted.key=value override")
    ap.add_argument("--eda", action="store_true")
    ap.add_argument("--predict", default="", help="run dir: rebuild model and write a test submission")
    a = ap.parse_args()
    cfg = merge(DEFAULTS, json.load(open(a.config)) if a.config else {})
    for kv in a.set:
        k, v = kv.split("=", 1)
        set_by_path(cfg, k, v)
    if a.eda:
        eda(resolve_paths(cfg))
    elif a.predict:
        print(repredict(a.predict, cfg["data_dir"] or None))
    else:
        run(cfg)

if __name__ == "__main__":
    main()

In [ ]:
import importlib, p2lner
for m in [m for m in list(sys.modules) if m.startswith("p2lner")]:
    del sys.modules[m]
from p2lner import data as D
from p2lner.train import run, resolve_ckpt, CKPT_FOLDERS, merge, set_by_path, DEFAULTS
from p2lner.run import eda
print("pipeline loaded")

## 3. SETTINGS — the only cell you edit

In [ ]:
RUN_LIST = ["E1_C_full_linear", "E3_A2_lora_bidir", "E4_D_adapter", "E5_KD_student_C4L"]
OVERRIDES = {}
SMOKE_TEST = False
RUN_EDA = True

EXPERIMENTS = {
    'E1_C_full_linear': {"backbone": {"ckpt": "@C", "attention": "bidirectional", "c_segment": 0}, "method": {"name": "full"}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 10, "batch_size": 8, "lr_backbone": 5e-05, "lr_new": 0.001}},
    'E2_B_lorma': {"backbone": {"ckpt": "@B", "attention": "bidirectional"}, "method": {"name": "lorma", "variant": "plus", "r": 8, "alpha": 16, "targets": ["attn", "ffn"]}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 15, "batch_size": 8, "lr_new": 0.001}},
    'E3_A2_lora_bidir': {"backbone": {"ckpt": "@A2", "attention": "bidirectional"}, "method": {"name": "lora", "r": 8, "alpha": 16, "dropout": 0.05, "targets": ["attn", "ffn"]}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 15, "batch_size": 8, "lr_new": 0.001}},
    'E3b_A2_lora_causal': {"backbone": {"ckpt": "@A2", "attention": "causal"}, "method": {"name": "lora", "r": 8, "alpha": 16, "dropout": 0.05, "targets": ["attn", "ffn"]}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 256}, "train": {"epochs": 15, "batch_size": 8, "lr_new": 0.001}},
    'E4_D_adapter': {"backbone": {"ckpt": "@D", "attention": "bidirectional"}, "method": {"name": "adapter", "bottleneck": 64}, "head": {"type": "linear"}, "window": {"max_len": 1024, "stride": 768}, "train": {"epochs": 15, "batch_size": 4, "lr_new": 0.001}},
    'E4b_D_bitfit': {"backbone": {"ckpt": "@D", "attention": "bidirectional"}, "method": {"name": "bitfit"}, "head": {"type": "linear"}, "window": {"max_len": 1024, "stride": 768}, "train": {"epochs": 15, "batch_size": 4, "lr_new": 0.002}},
    'E5_KD_student_C4L': {"backbone": {"ckpt": "@C", "attention": "bidirectional", "keep_layers": 4}, "method": {"name": "full"}, "head": {"type": "linear"}, "kd": {"teacher_run": "runs/E1_C_full_linear", "kd_alpha": 0.5, "kd_T": 2.0}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 12, "batch_size": 8, "lr_backbone": 0.0001, "lr_new": 0.001}},
    'E5c_noKD_student_C4L': {"backbone": {"ckpt": "@C", "attention": "bidirectional", "keep_layers": 4}, "method": {"name": "full"}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 12, "batch_size": 8, "lr_backbone": 0.0001, "lr_new": 0.001}},
    'E5b_KD_student_B': {"backbone": {"ckpt": "@B", "attention": "bidirectional"}, "method": {"name": "full"}, "head": {"type": "linear"}, "kd": {"teacher_run": "runs/E1_C_full_linear", "kd_alpha": 0.5, "kd_T": 2.0}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 12, "batch_size": 8, "lr_backbone": 0.0001, "lr_new": 0.001}},
    'E6_C_full_crf': {"backbone": {"ckpt": "@C", "attention": "bidirectional"}, "method": {"name": "full"}, "head": {"type": "crf"}, "loss": {"class_weight": "none"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 10, "batch_size": 8, "lr_backbone": 5e-05, "lr_new": 0.001}},
    'E6b_C_full_linear_cw': {"backbone": {"ckpt": "@C", "attention": "bidirectional"}, "method": {"name": "full"}, "head": {"type": "linear"}, "loss": {"class_weight": "sqrt_inv"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 10, "batch_size": 8, "lr_backbone": 5e-05, "lr_new": 0.001}},
    'E2b_C_lorma_pi': {"backbone": {"ckpt": "@C", "attention": "bidirectional"}, "method": {"name": "lorma", "variant": "pi", "r": 8, "targets": ["attn"]}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 15, "batch_size": 8, "lr_new": 0.001}},
    'X_C_frozen_probe': {"backbone": {"ckpt": "@C", "attention": "bidirectional"}, "method": {"name": "frozen"}, "head": {"type": "linear", "hidden": 256}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 10, "batch_size": 8, "lr_new": 0.002}},
    'X_A2_partial4': {"backbone": {"ckpt": "@A2", "attention": "bidirectional"}, "method": {"name": "partial", "train_last_k": 4}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 10, "batch_size": 8, "lr_backbone": 0.0001, "lr_new": 0.001}},
    'X_C_prompt': {"backbone": {"ckpt": "@C", "attention": "bidirectional"}, "method": {"name": "prompt", "n_prompt": 20}, "head": {"type": "linear"}, "window": {"max_len": 512, "stride": 384}, "train": {"epochs": 15, "batch_size": 8, "lr_new": 0.003}}
}
OUT_DIR = f"{WORK}/runs"

## 4. Data + checkpoint check

In [ ]:
DATA_DIR = D.find_data_dir([os.environ.get("P2_DATA_DIR", ""), f"{WORK}/p2data/Phase_2_LNER"])
if DATA_DIR is None:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("could not read Kaggle secret HF_TOKEN:", e)
    DATA_DIR = D.download(f"{WORK}/p2data")
print("data:", DATA_DIR)

CKPT_ROOT = os.environ.get("P2_CKPT_ROOT", "")
FOUND = {}
for key, folder in CKPT_FOLDERS.items():
    try:
        FOUND[key] = resolve_ckpt("@" + key, [CKPT_ROOT] if CKPT_ROOT else [])
        print(f"  @{key:3s} OK      {FOUND[key]}")
    except FileNotFoundError:
        print(f"  @{key:3s} MISSING {folder}/model_final.pt  -> runs using it will be skipped")

In [ ]:
if RUN_EDA:
    eda(merge(DEFAULTS, {"data_dir": str(DATA_DIR), "backbone": {"ckpt": FOUND.get("C") or next(iter(FOUND.values()))}}))

## 5. Train + evaluate + write submissions

In [ ]:
import time, traceback, csv
results = []
for name in RUN_LIST:
    cfg = json.loads(json.dumps(EXPERIMENTS[name]))
    cfg.update(run_name=name + ("_smoke" if SMOKE_TEST else ""), data_dir=str(DATA_DIR), out_dir=OUT_DIR, ckpt_root=CKPT_ROOT)
    key = cfg["backbone"]["ckpt"].lstrip("@")
    if key not in FOUND:
        print(f"\n##### SKIP {name}: checkpoint @{key} not attached"); continue
    if cfg.get("kd", {}).get("teacher_run"):
        t = os.path.join(OUT_DIR, os.path.basename(cfg["kd"]["teacher_run"]) + ("_smoke" if SMOKE_TEST else ""))
        if not os.path.exists(os.path.join(t, "best.pt")):
            print(f"\n##### SKIP {name}: teacher run {t} not found (run it earlier in RUN_LIST)"); continue
        cfg["kd"]["teacher_run"] = t
    for k, v in OVERRIDES.items():
        set_by_path(cfg, k, json.dumps(v))
    if SMOKE_TEST:
        cfg["backbone"]["keep_layers"] = 2; cfg["train"]["epochs"] = 1
    print(f"\n##################### {cfg['run_name']} #####################")
    try:
        summary, model = run(cfg)
        del model; torch.cuda.empty_cache()
        os.makedirs(f"{WORK}/submissions", exist_ok=True)
        dst = f"{WORK}/submissions/{cfg['run_name']}.zip"
        shutil.copy(summary["submission"], dst)
        results.append(summary)
        print(f">>> {cfg['run_name']}: dev macro-F1 {summary['dev']['macro_f1']:.4f} | submission {dst}")
    except Exception:
        traceback.print_exc()
        print(f">>> {name} FAILED (see traceback); continuing with the next run")

## 6. Results table (also saved to `/kaggle/working/RESULTS.csv`)

In [ ]:
rows = []
for p in sorted(glob.glob(f"{OUT_DIR}/*/summary.json")):
    s = json.load(open(p))
    rows.append({"run": s["run"], "arch": s["arch"], "method": s["method"]["name"], "head": s["head"], "attention": s["attention"],
                 "params_M": round(s["params_total"] / 1e6, 1), "trainable_M": round(s["params_trainable"] / 1e6, 3),
                 "trainable_%": s["trainable_pct"], "train_min": s["train_minutes"],
                 "peak_mem_GB": round(s["peak_train_mem_gb"] or 0, 2), "tok_per_s": s.get("infer_tokens_per_sec"),
                 "dev_macroF1": round(s.get("dev", {}).get("macro_f1", 0), 4), "dev_microF1": round(s.get("dev", {}).get("micro_f1", 0), 4),
                 "bio_ceiling": round(s["bio_ceiling_macro_f1"], 4)})
if rows:
    with open(f"{WORK}/RESULTS.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    try:
        import pandas as pd; display(pd.DataFrame(rows))
    except Exception:
        for r in rows: print(r)

## 7. Error analysis of one run (per-label scores, boundary vs type errors, examples)

In [ ]:
RUN = results[-1]["run"] if results else (rows[-1]["run"] if rows else None)
if RUN:
    from p2lner.metrics import format_table
    import matplotlib.pyplot as plt
    print(format_table(json.load(open(f"{OUT_DIR}/{RUN}/dev_scores.json"))))
    err = json.load(open(f"{OUT_DIR}/{RUN}/dev_errors.json"))
    print("\nerror categories:", err["categories"]); print("type confusions:", err["type_confusion"]); print("boundary errors:", err["boundary_kind"])
    for c, ex in err["examples"].items():
        print(f"\n-- {c}"); [print("   ", e) for e in ex]
    h = json.load(open(f"{OUT_DIR}/{RUN}/history.json"))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].plot([r["epoch"] for r in h], [r["train_loss"] for r in h], "o-"); ax[0].set_title(f"{RUN}: train loss")
    ax[1].plot([r["epoch"] for r in h if "dev_macro_f1" in r], [r["dev_macro_f1"] for r in h if "dev_macro_f1" in r], "o-")
    ax[1].set_title("dev strict macro-F1"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{RUN}/curves.png", dpi=120); plt.show()

## 8. Download